In [1]:
# ============================================================
# MultiEURLEX
# Load saved Parquet files
# + Extract 10,000 common CELEX train documents
# + Separate 4,604 adaptation pool
# + 4 verification checks
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict


# ============================================================
# 0. 설정
# ============================================================

DATA_PATH = "/content/drive/MyDrive/data"

langs = ["en", "de", "fr", "pl", "bg", "el", "fi", "mt"]

SEED = 42
N_TRAIN = 10_000


# ============================================================
# 1. 저장해둔 Parquet 파일 불러오기
# ============================================================

print("=" * 60)
print("1. Loading saved Parquet files")
print("=" * 60)

ds = {}

for l in langs:

    ds[l] = {}

    for split in ["train", "validation", "test"]:

        file_path = f"{DATA_PATH}/{l}_{split}.parquet"

        assert os.path.exists(file_path), (
            f"File not found: {file_path}"
        )

        df = pd.read_parquet(file_path)

        ds[l][split] = Dataset.from_pandas(
            df,
            preserve_index=False
        )

        print(
            f"{l}_{split}: "
            f"{len(ds[l][split]):,} rows"
        )

print("\n✅ All datasets loaded successfully.")


# ============================================================
# 2. Common Train CELEX 찾기
# ============================================================

print()
print("=" * 60)
print("2. Finding common Train CELEX IDs")
print("=" * 60)

train_celex_sets = {
    l: set(ds[l]["train"]["celex_id"])
    for l in langs
}

common_train_ids = sorted(
    set.intersection(*train_celex_sets.values())
)

print("Common train CELEX count:", len(common_train_ids))

assert len(common_train_ids) == 14_604

print("✅ Common train count = 14,604")


# ============================================================
# 3. Random 10,000개 추출
# ============================================================

print()
print("=" * 60)
print("3. Sampling 10,000 common training documents")
print("=" * 60)

rng = np.random.default_rng(SEED)

sample_indices = rng.choice(
    len(common_train_ids),
    size=N_TRAIN,
    replace=False
)

common_train_10k_ids = sorted(
    [common_train_ids[i] for i in sample_indices]
)

common_train_10k_set = set(common_train_10k_ids)

print("Random seed:", SEED)
print("Selected documents:", len(common_train_10k_ids))

assert len(common_train_10k_ids) == 10_000


# ============================================================
# 4. 나머지 4,604개를 adaptation pool로 분리
# ============================================================

adaptation_pool_ids = sorted(
    set(common_train_ids) - common_train_10k_set
)

print()
print("Main training set :", len(common_train_10k_ids))
print("Adaptation pool    :", len(adaptation_pool_ids))

assert len(adaptation_pool_ids) == 4_604

assert (
    set(common_train_10k_ids).isdisjoint(
        adaptation_pool_ids
    )
)

assert (
    set(common_train_10k_ids)
    | set(adaptation_pool_ids)
) == set(common_train_ids)

print("✅ 10,000 + 4,604 = 14,604")


# ============================================================
# 5. 각 언어에서 실제 Dataset 만들기
# ============================================================

print()
print("=" * 60)
print("4. Creating aligned datasets")
print("=" * 60)

train10k = {}
adaptation_pool = {}

for l in langs:

    # CELEX → row index
    id_to_idx = {
        cid: i
        for i, cid in enumerate(
            ds[l]["train"]["celex_id"]
        )
    }

    # 10,000 main training rows
    train_indices = [
        id_to_idx[cid]
        for cid in common_train_10k_ids
    ]

    # 4,604 adaptation rows
    adaptation_indices = [
        id_to_idx[cid]
        for cid in adaptation_pool_ids
    ]

    train10k[l] = ds[l]["train"].select(
        train_indices
    )

    adaptation_pool[l] = ds[l]["train"].select(
        adaptation_indices
    )

    print(
        f"{l}: "
        f"train10k={len(train10k[l]):,}, "
        f"adaptation={len(adaptation_pool[l]):,}"
    )


# ============================================================
# VERIFICATION 1
# 각 언어가 정확히 10,000개인지
# ============================================================

print()
print("=" * 60)
print("VERIFICATION 1 — Main train size")
print("=" * 60)

for l in langs:
    print(
        f"{l}: {len(train10k[l]):,}"
    )

assert all(
    len(train10k[l]) == 10_000
    for l in langs
)

print("✅ PASS")


# ============================================================
# VERIFICATION 2
# 8개 언어의 CELEX ID가 완전히 동일한지
# ============================================================

print()
print("=" * 60)
print("VERIFICATION 2 — CELEX alignment")
print("=" * 60)

train10k_celex_sets = {
    l: set(train10k[l]["celex_id"])
    for l in langs
}

first_lang = langs[0]

same_celex = all(
    train10k_celex_sets[l]
    == train10k_celex_sets[first_lang]
    for l in langs
)

print(
    "Same CELEX set across all 8 languages:",
    same_celex
)

assert same_celex

print("✅ PASS")


# ============================================================
# VERIFICATION 3
# adaptation pool이 정확히 4,604개인지
# ============================================================

print()
print("=" * 60)
print("VERIFICATION 3 — Adaptation pool size")
print("=" * 60)

for l in langs:
    print(
        f"{l}: {len(adaptation_pool[l]):,}"
    )

assert all(
    len(adaptation_pool[l]) == 4_604
    for l in langs
)

print("✅ PASS")


# ============================================================
# VERIFICATION 4
# Train / Validation / Test leakage 확인
# ============================================================

print()
print("=" * 60)
print("VERIFICATION 4 — Train/Val/Test leakage")
print("=" * 60)

main_train_set = set(common_train_10k_ids)

validation_sets = {
    l: set(ds[l]["validation"]["celex_id"])
    for l in langs
}

test_sets = {
    l: set(ds[l]["test"]["celex_id"])
    for l in langs
}

common_validation_ids = set.intersection(
    *validation_sets.values()
)

common_test_ids = set.intersection(
    *test_sets.values()
)

train_val_overlap = (
    main_train_set & common_validation_ids
)

train_test_overlap = (
    main_train_set & common_test_ids
)

val_test_overlap = (
    common_validation_ids & common_test_ids
)

print(
    "Main train ∩ validation:",
    len(train_val_overlap)
)

print(
    "Main train ∩ test:",
    len(train_test_overlap)
)

print(
    "Validation ∩ test:",
    len(val_test_overlap)
)

assert len(train_val_overlap) == 0
assert len(train_test_overlap) == 0
assert len(val_test_overlap) == 0

print("✅ PASS")


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 60)
print("FINAL DATA SPLIT")
print("=" * 60)

print(
    f"Common train pool : {len(common_train_ids):,}"
)

print(
    f"Main train        : {len(common_train_10k_ids):,}"
)

print(
    f"Adaptation pool   : {len(adaptation_pool_ids):,}"
)

print(
    f"Validation        : {len(common_validation_ids):,}"
)

print(
    f"Test              : {len(common_test_ids):,}"
)

print()
print("Random seed:", SEED)
print("Languages:", ", ".join(langs))

print()
print("🎉 ALL 4 VERIFICATION CHECKS PASSED")

Mounted at /content/drive
1. Loading saved Parquet files
en_train: 55,000 rows
en_validation: 5,000 rows
en_test: 5,000 rows
de_train: 55,000 rows
de_validation: 5,000 rows
de_test: 5,000 rows
fr_train: 55,000 rows
fr_validation: 5,000 rows
fr_test: 5,000 rows
pl_train: 23,197 rows
pl_validation: 5,000 rows
pl_test: 5,000 rows
bg_train: 15,986 rows
bg_validation: 5,000 rows
bg_test: 5,000 rows
el_train: 55,000 rows
el_validation: 5,000 rows
el_test: 5,000 rows
fi_train: 42,497 rows
fi_validation: 5,000 rows
fi_test: 5,000 rows
mt_train: 17,521 rows
mt_validation: 5,000 rows
mt_test: 5,000 rows

✅ All datasets loaded successfully.

2. Finding common Train CELEX IDs
Common train CELEX count: 14604
✅ Common train count = 14,604

3. Sampling 10,000 common training documents
Random seed: 42
Selected documents: 10000

Main training set : 10000
Adaptation pool    : 4604
✅ 10,000 + 4,604 = 14,604

4. Creating aligned datasets
en: train10k=10,000, adaptation=4,604
de: train10k=10,000, adaptatio

In [2]:
# ============================================================
# Label distribution check
# 14,604 common train vs 10,000 sampled train
# ============================================================

from collections import Counter
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Multi-label frequency 계산 함수
# ------------------------------------------------------------

def get_label_counts(dataset):
    counter = Counter()

    for labels in dataset["labels"]:
        counter.update(labels)

    return counter


# ------------------------------------------------------------
# 2. EN 기준으로 원본 14,604개와 10,000개 비교
# ------------------------------------------------------------

# Common 14,604 documents
common_train_dataset = ds["en"]["train"].filter(
    lambda x: x["celex_id"] in set(common_train_ids)
)

full_counts = get_label_counts(common_train_dataset)
sample_counts = get_label_counts(train10k["en"])

num_full = len(common_train_dataset)
num_sample = len(train10k["en"])

# ------------------------------------------------------------
# 3. 비교표 생성
# ------------------------------------------------------------

rows = []

for label_id in range(21):

    full_count = full_counts[label_id]
    sample_count = sample_counts[label_id]

    full_rate = full_count / num_full
    sample_rate = sample_count / num_sample

    diff_pp = (sample_rate - full_rate) * 100

    rows.append({
        "label_id": label_id,
        "full_count_14604": full_count,
        "full_rate_%": full_rate * 100,
        "sample_count_10000": sample_count,
        "sample_rate_%": sample_rate * 100,
        "difference_pp": diff_pp
    })

label_dist = pd.DataFrame(rows)

label_dist = label_dist.sort_values(
    "label_id"
).reset_index(drop=True)


# ------------------------------------------------------------
# 4. 결과 출력
# ------------------------------------------------------------

print("=" * 80)
print("LABEL DISTRIBUTION: 14,604 vs 10,000")
print("=" * 80)

display(
    label_dist.round({
        "full_rate_%": 3,
        "sample_rate_%": 3,
        "difference_pp": 3
    })
)


# ------------------------------------------------------------
# 5. 가장 큰 차이 확인
# ------------------------------------------------------------

max_abs_diff = label_dist["difference_pp"].abs().max()

print()
print(
    f"Maximum absolute difference: "
    f"{max_abs_diff:.3f} percentage points"
)

print()
print("Labels with largest differences:")

display(
    label_dist
    .sort_values(
        "difference_pp",
        key=lambda x: x.abs(),
        ascending=False
    )
    .head(5)
    .round(3)
)


# ------------------------------------------------------------
# 6. 8개 언어에서 10,000개 label distribution이 동일한지
# ------------------------------------------------------------

print()
print("=" * 80)
print("CROSS-LANGUAGE LABEL DISTRIBUTION CHECK")
print("=" * 80)

lang_counts = {}

for l in langs:
    lang_counts[l] = get_label_counts(train10k[l])

same_distribution = True

for label_id in range(21):

    counts = [
        lang_counts[l][label_id]
        for l in langs
    ]

    if len(set(counts)) != 1:
        same_distribution = False
        print(
            f"Label {label_id}: "
            f"{dict(zip(langs, counts))}"
        )

print(
    "Same label counts across all 8 languages:",
    same_distribution
)

assert same_distribution

print("✅ PASS")

Filter:   0%|          | 0/55000 [00:00<?, ? examples/s]

LABEL DISTRIBUTION: 14,604 vs 10,000


,label_id,full_count_14604,full_rate_%,sample_count_10000,sample_rate_%,difference_pp
0,0,3010,20.611,2032,20.32,-0.291
1,1,1668,11.422,1145,11.45,0.028
2,2,2049,14.030,1392,13.92,-0.110
3,3,7320,50.123,5018,50.18,0.057
4,4,1417,9.703,974,9.74,0.037
5,5,2313,15.838,1578,15.78,-0.058
6,6,5743,39.325,3955,39.55,0.225
7,7,1954,13.380,1309,13.09,-0.290
8,8,1484,10.162,1008,10.08,-0.082
9,9,743,5.088,516,5.16,0.072



Maximum absolute difference: 0.445 percentage points

Labels with largest differences:


,label_id,full_count_14604,full_rate_%,sample_count_10000,sample_rate_%,difference_pp
18,18,4731,32.395,3284,32.84,0.445
0,0,3010,20.611,2032,20.32,-0.291
7,7,1954,13.380,1309,13.09,-0.290
17,17,5455,37.353,3762,37.62,0.267
6,6,5743,39.325,3955,39.55,0.225



CROSS-LANGUAGE LABEL DISTRIBUTION CHECK
Same label counts across all 8 languages: True
✅ PASS
